In [14]:
# ==========================================
# CELL 1: SETUP & GET OFFICIAL DICTIONARY
# ==========================================
import sys
!{sys.executable} -m pip -q install tensorflow tensorflow-hub librosa numpy pandas

import os
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/reWatt'
EXPORT_DIR = f'{ROOT}/exports/yamnet_tflite'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Download the official YAMNet class map (521 strings)
csv_url = 'https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/yamnet_class_map.csv'
class_map_df = pd.read_csv(csv_url)
# Extract just the display names as a list of strings
ALL_CLASS_NAMES = class_map_df['display_name'].tolist()

print(f"✅ Environment ready. Loaded {len(ALL_CLASS_NAMES)} official class names.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Environment ready. Loaded 521 official class names.


In [15]:
# ==========================================
# CELL 2: BUILD MULTI-LABEL MODULE
# ==========================================

class YAMNetMultiLabel(tf.Module):
    def __init__(self, class_names_list, top_k=5):
        super(YAMNetMultiLabel, self).__init__()
        print("Downloading base YAMNet model from TF Hub...")
        self.yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

        # Bake all 521 strings into memory
        self.class_names = tf.constant(class_names_list)

        # Define how many simultaneous labels we want to return
        self.top_k = tf.constant(top_k, dtype=tf.int32)

    @tf.function(input_signature=[tf.TensorSpec(shape=[None], dtype=tf.float32, name='audio_waveform')])
    def __call__(self, waveform):
        # 1. Run the raw audio through YAMNet
        scores, embeddings, spectrogram = self.yamnet(waveform)

        # 2. Extract peak probabilities across the entire clip
        peak_scores = tf.reduce_max(scores, axis=0)

        # 3. FIX: Get the Top K scores and their corresponding indices
        top_scores, top_indices = tf.math.top_k(peak_scores, k=self.top_k)

        # 4. Look up the official string names for those top indices
        top_class_names = tf.gather(self.class_names, top_indices)

        # Output arrays instead of single values!
        return {
            'top_class_names': top_class_names,
            'top_scores': top_scores,
            'all_521_scores': peak_scores
        }

print("Initializing Multi-Label Classifier module (Top-5)...")
multi_label_model = YAMNetMultiLabel(ALL_CLASS_NAMES, top_k=5)
print("✅ Module initialized.")

Initializing Multi-Label Classifier module (Top-5)...
✅ Module initialized.


In [19]:
# ==========================================
# CELL 3: TFLITE FLOAT16 EXPORT
# ==========================================

print("Tracing the execution graph...")
concrete_func = multi_label_model.__call__.get_concrete_function()

print("Converting to TFLite with Float16 Quantization...")
converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

# FIX: Changed 'universal' to 'multilabel' to match the model being exported!
export_path = f'{EXPORT_DIR}/yamnet_multilabel_fp16.tflite'
with open(export_path, 'wb') as f:
    f.write(tflite_model)

model_size_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f"🚀 Success! Model exported to: {export_path}")
print(f"📉 New Float16 Model Size: {model_size_mb:.2f} MB")

Tracing the execution graph...


Converting to TFLite with Float16 Quantization...
🚀 Success! Model exported to: /content/drive/MyDrive/reWatt/exports/yamnet_tflite/yamnet_multilabel_fp16.tflite
📉 New Float16 Model Size: 7.20 MB


In [21]:
# ==========================================
# CELL 4: VERIFY MULTI-LABEL INFERENCE
# ==========================================
import numpy as np
import tensorflow as tf

# 1. Load the TFLite model
interpreter = tf.lite.Interpreter(model_path=export_path)
input_details = interpreter.get_input_details()

# 2. Create 5 seconds of mock audio (dead silence)
dummy_audio = np.zeros(80000, dtype=np.float32)

# 3. Resize and allocate
interpreter.resize_tensor_input(input_details[0]['index'], dummy_audio.shape)
interpreter.allocate_tensors()
interpreter.set_tensor(input_details[0]['index'], dummy_audio)

# 4. Run inference
interpreter.invoke()

# 5. Extract the array outputs
output_details = interpreter.get_output_details()
results = {}

for detail in output_details:
    val = interpreter.get_tensor(detail['index'])

    # Grab the massive 521-length array
    if len(val.shape) > 0 and val.shape[-1] == 521:
        results['all_scores'] = val

    # Grab the Top-K Float Array (e.g., shape (5,))
    elif np.issubdtype(detail['dtype'], np.floating):
        results['top_scores'] = val

    # Grab the Top-K String Array (e.g., shape (5,))
    else:
        # Decode the array of bytes into a standard Python list of strings
        try:
            results['top_names'] = [item.decode('utf-8') for item in val]
        except AttributeError:
            results['top_names'] = [str(item) for item in val]

print("\n--- TEST INFERENCE RESULTS (TOP 5) ---")
print("Detected Sounds ranked by confidence:")

# Zip the names and scores together to print them nicely
for rank, (name, score) in enumerate(zip(results['top_names'], results['top_scores']), start=1):
    print(f" {rank}. {name:<20} | Confidence: {score:.4f}")

if "Silence" in results['top_names']:
    print("\n✅ Multi-Label Model parsed successfully!")


--- TEST INFERENCE RESULTS (TOP 5) ---
Detected Sounds ranked by confidence:
 1. Silence              | Confidence: 1.0000
 2. Speech               | Confidence: 0.0000
 3. Music                | Confidence: 0.0000
 4. Inside, small room   | Confidence: 0.0000
 5. Narration, monologue | Confidence: 0.0000

✅ Multi-Label Model parsed successfully!


In [22]:
# ==========================================
# CELL 4: TEST WITH REAL AUDIO (LIBROSA)
# ==========================================
import numpy as np
import tensorflow as tf
import librosa
import urllib.request
import os

# 1. Load the TFLite model
interpreter = tf.lite.Interpreter(model_path=export_path)
input_details = interpreter.get_input_details()

# 2. Download a quick test audio file (e.g., a dog barking)
test_audio_url = "https://storage.googleapis.com/audioset/miaow_16k.wav" # Actually a cat meowing!
test_file_path = "test_audio.wav"

if not os.path.exists(test_file_path):
    print("Downloading test audio...")
    urllib.request.urlretrieve(test_audio_url, test_file_path)

# 3. Load the audio with Librosa!
# WARNING: sr=16000 and mono=True are strictly required for YAMNet
print("Processing audio with librosa...")
waveform, sample_rate = librosa.load(test_file_path, sr=16000, mono=True)

# 4. Resize and allocate based on the actual audio length
interpreter.resize_tensor_input(input_details[0]['index'], waveform.shape)
interpreter.allocate_tensors()
interpreter.set_tensor(input_details[0]['index'], waveform)

# 5. Run inference
print("Running inference...")
interpreter.invoke()

# 6. Extract the array outputs
output_details = interpreter.get_output_details()
results = {}

for detail in output_details:
    val = interpreter.get_tensor(detail['index'])
    if len(val.shape) > 0 and val.shape[-1] == 521:
        results['all_scores'] = val
    elif np.issubdtype(detail['dtype'], np.floating):
        results['top_scores'] = val
    else:
        try:
            results['top_names'] = [item.decode('utf-8') for item in val]
        except AttributeError:
            results['top_names'] = [str(item) for item in val]

print("\n--- TEST INFERENCE RESULTS (TOP 5) ---")
print("Detected Sounds ranked by confidence:")

for rank, (name, score) in enumerate(zip(results['top_names'], results['top_scores']), start=1):
    print(f" {rank}. {name:<20} | Confidence: {score:.4f}")

Processing audio with librosa...
Running inference...

--- TEST INFERENCE RESULTS (TOP 5) ---
Detected Sounds ranked by confidence:
 1. Animal               | Confidence: 0.9747
 2. Cat                  | Confidence: 0.9474
 3. Domestic animals, pets | Confidence: 0.9245
 4. Meow                 | Confidence: 0.7844
 5. Chicken, rooster     | Confidence: 0.5574
